# LSTM v01 — shared selected node features

LSTM dùng chính xác node-feature contract đã chọn cho GAT-GRU. Tune cùng CPU-weighted macro validation objective và đánh giá cùng frozen test split.

In [1]:
import json
import sys
from pathlib import Path
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa

ctx = fa.load_context(ROOT)
selection = fa.load_selection_contract(
    ROOT / 'configs' / 'gatgru' / 'feature_contract_v01.json', require_edge=False)
base_cfg = json.loads((ROOT / 'dataset' / 'processed' / 'lstm' / 'lstm_best_config.json').read_text())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ROOT / 'artifacts' / 'lstm' / 'v01_shared_selected_features'
print('shared node features:', selection['node_features'])
print('device:', DEVICE)

shared node features: ['rps_in', 'rps_in_delta', 'rps_per_replica', 'latency_p50_ms', 'latency_p95_ms', 'cpu_cores', 'mem_mib', 'net_rx_kBps', 'net_tx_kBps', 'replicas']
device: cpu


## 1. Tune training hyperparameters with the LSTM architecture fixed

In [2]:
best_cfg, study = fa.run_optuna_fixed_architecture(
    ctx, 'lstm', selection['node_features'], [], base_cfg, OUT / 'tuning',
    n_trials=24, seed=42, device=DEVICE, score_weights=fa.CPU_DIAGNOSTIC_WEIGHTS,
    parent_version='lstm-baseline-2026-10-14',
)
print('best score:', study.best_value)
best_cfg

[I 2026-10-19 09:48:08,406] A new study created in RDB with name: lstm_b55e5fc117df47d5_fixed_arch
[I 2026-10-19 09:49:08,608] Trial 0 finished with value: 0.10371746689369507 and parameters: {'window': 18, 'dropout': 0.11973169683940732, 'lr': 0.0006427207987637223, 'weight_decay': 2.9375384576328313e-06, 'batch_size': 64}. Best is trial 0 with value: 0.10371746689369507.
[I 2026-10-19 09:50:35,611] Trial 1 finished with value: 0.1062993819073528 and parameters: {'window': 24, 'dropout': 0.16648852816008436, 'lr': 0.0007037018186923737, 'weight_decay': 3.5113563139704077e-06, 'batch_size': 128}. Best is trial 0 with value: 0.10371746689369507.
[I 2026-10-19 09:54:03,429] Trial 2 finished with value: 0.10470839158618886 and parameters: {'window': 24, 'dropout': 0.027898772130408367, 'lr': 0.0008001480489146253, 'weight_decay': 1.2562773503807034e-05, 'batch_size': 64}. Best is trial 0 with value: 0.10371746689369507.
[I 2026-10-19 09:55:24,581] Trial 3 finished with value: 0.1039428010

best score: 0.10101475275583169


{'window': 24,
 'dropout': 0.10338721424396377,
 'lr': 0.0011406112035118004,
 'batch_size': 32,
 'hidden': 128,
 'layers': 2,
 'mode': 'joint',
 'weight_decay': 6.4874484174886886e-06}

## 2. Five seeds and final test evaluation

In [3]:
prediction_path, training_meta = fa.run_final(
    ctx, 'lstm', selection['node_features'], [], best_cfg, OUT,
    seeds=(0, 1, 2, 3, 4), device=DEVICE, model_name='lstm_shared',
    parent_version='lstm-baseline-2026-10-14',
)
print(prediction_path)
training_meta

F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\lstm-load-testing\experiments\lstm\v01_shared_selected_features\lstm_shared_predictions_test.npz


{'model': 'lstm_shared',
 'model_kind': 'lstm',
 'config': {'window': 24,
  'dropout': 0.10338721424396377,
  'lr': 0.0011406112035118004,
  'batch_size': 32,
  'hidden': 128,
  'layers': 2,
  'mode': 'joint',
  'weight_decay': 6.4874484174886886e-06},
 'seeds': [0, 1, 2, 3, 4],
 'val_mae_scaled': [0.109454445540905,
  0.11036039888858795,
  0.10946721583604813,
  0.1099720448255539,
  0.11016689985990524],
 'node_features': ['rps_in',
  'rps_in_delta',
  'rps_per_replica',
  'latency_p50_ms',
  'latency_p95_ms',
  'cpu_cores',
  'mem_mib',
  'net_rx_kBps',
  'net_tx_kBps',
  'replicas'],
 'edge_features': [],
 'targets': ['rps_in', 'cpu_cores'],
 'services': ['frontend',
  'adservice',
  'cartservice',
  'checkoutservice',
  'currencyservice',
  'emailservice',
  'paymentservice',
  'productcatalogservice',
  'recommendationservice',
  'redis-cart',
  'shippingservice'],
 'n_params': 294576,
 'inference_ms': 1.1767499963752925,
 'scalers': {'node': {'clip': [61.065799713134766,
    14